# Módulo DS4 · Sistemas de recomendación

**Caso:** una plataforma de streaming quiere mostrar a cada usuario las películas que más probablemente le gusten. Tenemos una matriz de valoraciones (1–5 estrellas) muy **dispersa**: cada usuario ha valorado solo una pequeña parte del catálogo.

Objetivos:
1. Construir una referencia por **popularidad**.
2. Implementar **filtrado colaborativo** ítem-ítem con similitud del coseno.
3. Implementar **factorización de matrices** (ALS) para descubrir "gustos latentes".
4. Evaluar con RMSE (precisión de la nota) y precisión@k (calidad del ranking).

> Los datos se simulan con una estructura realista para que el notebook funcione sin descargas. En el ejercicio final repetirás el análisis con **MovieLens 100K**, el conjunto de referencia del sector.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(1)

## 1. Datos: 600 usuarios × 300 películas
Cada usuario y cada película tienen un vector de 4 "gustos" latentes (acción, comedia, drama, ciencia ficción). La nota depende de cuánto encajan, más sesgos de usuario (hay usuarios generosos) y de película (hay películas mejores).

In [ ]:
n_usuarios, n_peliculas, k_real = 600, 300, 4
U_real = rng.normal(0, 1, (n_usuarios, k_real))
V_real = rng.normal(0, 1, (n_peliculas, k_real))
sesgo_u = rng.normal(0, 0.4, n_usuarios)
sesgo_p = rng.normal(0, 0.6, n_peliculas)
nota_latente = 3.4 + sesgo_u[:, None] + sesgo_p[None, :] + 0.5 * U_real @ V_real.T + rng.normal(0, 0.4, (n_usuarios, n_peliculas))
notas_completas = np.clip(np.round(nota_latente), 1, 5)

# Los usuarios ven más las películas populares y las que creen que les gustarán
popularidad = rng.pareto(1.5, n_peliculas) + 1
prob_ver = popularidad[None, :] / popularidad.sum() * 30 * np.exp(0.3 * (nota_latente - 3))
observada = rng.random((n_usuarios, n_peliculas)) < np.clip(prob_ver, 0, 0.9)

filas, cols = np.where(observada)
valoraciones = pd.DataFrame({"usuario": filas, "pelicula": cols, "nota": notas_completas[filas, cols]})
print(f"{len(valoraciones):,} valoraciones | densidad de la matriz: {observada.mean():.1%}")
valoraciones.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
valoraciones["nota"].value_counts().sort_index().plot.bar(ax=axes[0], title="Distribución de notas")
valoraciones.groupby("pelicula").size().sort_values(ascending=False).reset_index(drop=True).plot(ax=axes[1], title="Valoraciones por película (cola larga)")
valoraciones.groupby("usuario").size().plot.hist(bins=40, ax=axes[2], title="Valoraciones por usuario")
plt.tight_layout(); plt.show()

## 2. Partición train/test
Para cada usuario reservamos el 20 % de sus valoraciones como test.

In [ ]:
test = valoraciones.groupby("usuario", group_keys=False).sample(frac=0.2, random_state=0)
train = valoraciones.drop(test.index)

R = np.full((n_usuarios, n_peliculas), np.nan)
R[train["usuario"], train["pelicula"]] = train["nota"]
media_global = np.nanmean(R)

def rmse(pred, df=test):
    return np.sqrt(np.mean((pred[df["usuario"], df["pelicula"]] - df["nota"]) ** 2))

def precision_at_k(puntuaciones, k=5, umbral=5):
    """Ordenamos las películas de test de cada usuario según el modelo y tomamos las k primeras:
    ¿qué fracción le encantó (nota ≥ umbral)?"""
    precisiones = []
    for u, grupo in test.groupby("usuario"):
        if len(grupo) < k:
            continue
        top = grupo.assign(p=puntuaciones[u, grupo["pelicula"]]).nlargest(k, "p")
        precisiones.append((top["nota"] >= umbral).mean())
    return np.mean(precisiones)

## 3. Referencias simples
- **Media global**: misma nota para todo.
- **Sesgos**: media global + sesgo del usuario + sesgo de la película (con regularización para películas con pocas notas).

In [ ]:
pred_global = np.full((n_usuarios, n_peliculas), media_global)

def sesgos(R, lam=5, iters=10):
    bu, bi = np.zeros(R.shape[0]), np.zeros(R.shape[1])
    mask = ~np.isnan(R)
    Rc = np.where(mask, R - media_global, 0)
    for _ in range(iters):
        bi = ((Rc - bu[:, None]) * mask).sum(0) / (mask.sum(0) + lam)
        bu = ((Rc - bi[None, :]) * mask).sum(1) / (mask.sum(1) + lam)
    return bu, bi

bu, bi = sesgos(R)
pred_sesgos = media_global + bu[:, None] + bi[None, :]

# Popularidad: puntuar cada película por su número de valoraciones (igual para todos)
pred_popular = np.tile(np.bincount(train["pelicula"], minlength=n_peliculas), (n_usuarios, 1)).astype(float)

## 4. Filtrado colaborativo ítem-ítem
"A quien le gustó X también le gustó Y". Dos películas son similares si los usuarios las valoran de forma parecida (tras restar los sesgos). Predecimos la nota de un usuario para una película como la media ponderada de sus notas a las películas **más similares** que ya ha visto.

In [ ]:
def filtrado_item_item(R, k_vecinos=30, encogimiento=10):
    mask = ~np.isnan(R)
    residuos = np.where(mask, R - pred_sesgos, 0)
    co = mask.T.astype(float) @ mask.astype(float)                  # usuarios en común
    normas = np.sqrt((residuos ** 2).sum(0))
    sim = (residuos.T @ residuos) / (np.outer(normas, normas) + 1e-9)
    sim *= co / (co + encogimiento)                                  # desconfía de pares con pocos usuarios en común
    np.fill_diagonal(sim, 0)
    # quedarnos con los k vecinos más similares de cada película
    corte = -np.sort(-sim, axis=1)[:, k_vecinos - 1:k_vecinos]
    sim = np.where(sim >= corte, sim, 0)
    num = residuos @ sim.T
    den = mask.astype(float) @ np.abs(sim).T
    return pred_sesgos + np.divide(num, den, out=np.zeros_like(num), where=den > 1e-9), sim

pred_item, sim_items = filtrado_item_item(R)

## 5. Factorización de matrices (ALS)
Aproximamos la matriz de residuos por el producto de dos matrices pequeñas: $R \approx \mu + b_u + b_i + P Q^\top$, con $P$ (usuarios × *k*) y $Q$ (películas × *k*). Cada fila es un vector de gustos latentes. **ALS** alterna: fija $Q$ y resuelve $P$ por mínimos cuadrados regularizados, luego al revés.

In [ ]:
def als(R, k=8, lam=8.0, iters=15, rng=rng):
    mask = ~np.isnan(R)
    Rc = np.where(mask, R - pred_sesgos, 0)
    P = rng.normal(0, 0.1, (R.shape[0], k))
    Q = rng.normal(0, 0.1, (R.shape[1], k))
    I = lam * np.eye(k)
    for _ in range(iters):
        for u in range(R.shape[0]):
            idx = mask[u]
            P[u] = np.linalg.solve(Q[idx].T @ Q[idx] + I, Q[idx].T @ Rc[u, idx])
        for i in range(R.shape[1]):
            idx = mask[:, i]
            Q[i] = np.linalg.solve(P[idx].T @ P[idx] + I, P[idx].T @ Rc[idx, i])
    return pred_sesgos + P @ Q.T, P, Q

pred_als, P, Q = als(R)

## 6. Comparación

In [ ]:
resultados = pd.DataFrame({
    "RMSE": [np.nan, rmse(pred_global), rmse(pred_sesgos), rmse(pred_item), rmse(pred_als)],
    "precisión@5": [precision_at_k(pred_popular), precision_at_k(pred_global + rng.normal(0, 1e-6, pred_global.shape)),
                     precision_at_k(pred_sesgos), precision_at_k(pred_item), precision_at_k(pred_als)],
}, index=["Popularidad", "Media global (≈ aleatorio)", "Sesgos", "Ítem-ítem", "ALS (k=8)"])
resultados.round(3)

- La **popularidad** no predice notas y aquí apenas supera al azar, porque solo ordena películas que el usuario ya ha visto. Su valor real está en el **arranque en frío**: es lo único que puedes mostrar a un usuario sin historial.
- Los modelos personalizados mejoran tanto el error como el ranking: ALS acierta más a menudo con las películas que el usuario puntuará con 5 estrellas.

## 7. ¿Cuántos factores latentes?

In [ ]:
ks = [1, 2, 4, 8, 16, 32]
errores = [rmse(als(R, k=k, iters=8)[0]) for k in ks]
plt.plot(ks, errores, "o-"); plt.xscale("log", base=2)
plt.xlabel("k (factores latentes)"); plt.ylabel("RMSE test"); plt.title("Selección de k"); plt.show()

## 8. Recomendaciones para un usuario y películas similares

In [ ]:
u = 42
vistas = set(train.loc[train["usuario"] == u, "pelicula"])
candidatas = [i for i in np.argsort(-pred_als[u]) if i not in vistas][:10]
print(f"Usuario {u} – top 10 recomendaciones:", [int(i) for i in candidatas])

p = int(train["pelicula"].value_counts().index[0])
print(f"Películas más similares a la {p} (ítem-ítem):", [int(i) for i in np.argsort(-sim_items[p])[:5]])

## Ejercicios

1. **Arranque en frío:** ¿qué recomendarías a un usuario recién registrado? ¿Y cómo tratarías una película recién estrenada sin valoraciones? Propón una estrategia híbrida.
2. Varía `lam` en ALS (0,1; 1; 8; 30). Explica el resultado en términos de sobreajuste.
3. Implementa el filtrado colaborativo **usuario-usuario** y compáralo con ítem-ítem. ¿Cuál es más barato de calcular si tienes 10 millones de usuarios y 50.000 productos?
4. **Datos reales:** descarga MovieLens 100K
   ```python
   url = "https://files.grouplens.org/datasets/movielens/ml-100k/u.data"
   df = pd.read_csv(url, sep="\t", names=["usuario", "pelicula", "nota", "ts"])
   ```
   y repite la comparación de la sección 6 (recuerda que los IDs empiezan en 1).

In [ ]:
# TU CÓDIGO AQUÍ